In [ ]:
# import os
# os.environ['LD_PRELOAD'] = '/usr/lib64/libstdc++.so.6'
import sdot
import torch


def close( a, b, eps = 1e-5 ):
    return abs( a - b ).max() < eps

def enable_grad_for_attr(obj, attr_name):
    """Helper: clone/detach un attribut pour en faire un leaf tensor avec grad"""
    tensor = getattr(obj, attr_name)
    if tensor is not None:
        tensor = tensor.clone().detach().requires_grad_(True)
        setattr(obj, attr_name, tensor)
    return tensor

In [6]:
sdot.driver.framework = "torch"
sdot.driver.device = "cpu"      # ou "cuda" si dispo
sdot.driver.dtype = "FP64"      # FP32 possible

name_ot = "[ 0, 1 ] => 1"
f= sdot.SumOfWeightedDiracs1d([0.0, 1.0])
g = sdot.PiecewiseAffineFunction1d([1.0, 1.0])
exp_dist = [1 / 12]
exp_bary = [1 / 4, 3 / 4]

In [10]:
print("name of optimal transport:", name_ot)
plan = sdot.plan(f, g)

assert close(plan.distances, sdot.driver.t1(exp_dist)),  f"bad distances"
assert close(plan.barycenters, sdot.driver.t1(exp_bary)), f"bad barycenters"
print("grad f.positions:", f.positions.grad)
print("grad f.weights:", f.weights.grad)
print("grad g.ys:", g.ys.grad)
print("grad g.xs:", g.xs.grad)

name of optimal transport: [ 0, 1 ] => 1
grad f.positions: None
grad f.weights: None
grad g.ys: None
grad g.xs: None


In [11]:
f.positions.requires_grad_(True)
f.weights.requires_grad_(True)
g.ys.requires_grad_(True)
g.xs.requires_grad_(True)
print("Après activation des gradients:")
print("f.positions.requires_grad:", f.positions.requires_grad)
print("f.weights.requires_grad:", f.weights.requires_grad)
print("g.ys.requires_grad:", g.ys.requires_grad)
print("g.xs.requires_grad:", g.xs.requires_grad)

Après activation des gradients:
f.positions.requires_grad: True
f.weights.requires_grad: False
g.ys.requires_grad: True
g.xs.requires_grad: False


# Attention : Binding et Autograd PyTorch

> **Note Importante :**
> `f.weights` est retourné via un getter (une propriété) → peut être un tenseur intermédiaire.
> Les bindings C++ recopient/transforment les données → le tenseur original n'est pas dans le graphe d'autograd.
>  d'apèrs Grok Code Fast 1

## Comportement d'Autograd de PyTorch

PyTorch n'attribue `.grad` qu'aux **leaf tensors** (racines du graphe).

Pour s'assurer que `f.weights` est bien un **leaf tensor** et qu'il peut recevoir un gradient, utilisez la séquence suivante :

```python
f.weights = f.weights.clone().detach().requires_grad_(True)



In [31]:

enable_grad_for_attr(f, "positions")
enable_grad_for_attr(f, "weights")
enable_grad_for_attr(g, "ys")
enable_grad_for_attr(g, "xs")
print("Après activation des gradients:")
print("f.positions.requires_grad:", f.positions.requires_grad)
print("f.weights.requires_grad:", f.weights.requires_grad)
print("g.ys.requires_grad:", g.ys.requires_grad)
print("g.xs.requires_grad:", g.xs.requires_grad)

Après activation des gradients:
f.positions.requires_grad: True
f.weights.requires_grad: True
g.ys.requires_grad: True
g.xs.requires_grad: True


In [32]:
# Loss + backward
loss = sdot.distances(f, g).sum()
loss.backward()

print("✅ Backward OK")
print("  grad f.positions:", f.positions.grad)
print("  grad f.weights:", f.weights.grad)
print("  grad g.ys:", g.ys.grad)
print("  grad g.xs:", g.xs.grad)

✅ Backward OK
  grad f.positions: tensor([-0.2500,  0.2500], dtype=torch.float64)
  grad f.weights: tensor([0., 0.], dtype=torch.float64)
  grad g.ys: tensor([0., 0.], dtype=torch.float64)
  grad g.xs: tensor([ 0.0833, -0.0833], dtype=torch.float64)


In [35]:
optimizer = torch.optim.LBFGS([f.positions, f.weights, g.ys], lr=0.1, max_iter=20)
def closure():
    optimizer.zero_grad()
    plan = sdot.plan(f, g)
    loss = sdot.distances(f, g).sum()
    loss.backward()
    return loss

for i in range(10):
    loss = optimizer.step(closure)
    print(f"iter {i}: loss = {loss.item():.5f}")

print("✅ Optimisation terminée")
print("final f.positions:", f.positions.detach().numpy())
print("final f.weights:", f.weights.detach().numpy())
print("final g.ys:", g.ys.detach().numpy())
print("final g.xs:", g.xs.detach().numpy())
#


iter 0: loss = 0.02083
iter 1: loss = 0.02083
iter 2: loss = 0.02083
iter 3: loss = 0.02083
iter 4: loss = 0.02083
iter 5: loss = 0.02083
iter 6: loss = 0.02083
iter 7: loss = 0.02083
iter 8: loss = 0.02083
iter 9: loss = 0.02083
✅ Optimisation terminée
final f.positions: [0.24997884 0.75002116]
final f.weights: [1. 1.]
final g.ys: [1. 1.]
final g.xs: [0. 1.]


# 📊 Comparaison de qq optimiseurs ici
   Méthode      | Avantage                     | Inconvénient                     |
 |--------------|------------------------------|----------------------------------|
 | Manuelle     | Simple, transparent           | À réinventer à chaque fois       |
 | Adam         | Rapide, adaptatif            | Peut être moins stable          |
 | LBFGS        | Très efficace pour petits problèmes | Plus lent par itération |